<a href="https://colab.research.google.com/github/Jolieyolie/AI-agent/blob/main/Copy_of_Week3_AG_News_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment: AG News Topic Classifier

## Welcome!

In this assignment you will build a **news topic classifier** by fine-tuning a pre-trained **DistilBERT** model on the [AG News](https://huggingface.co/datasets/ag_news) dataset.

AG News is a standard NLP benchmark that contains short news articles grouped into four topics:

| Label | Category |
|-------|----------|
| 0     | World    |
| 1     | Sports   |
| 2     | Business |
| 3     | Sci/Tech |

Your task is to build a pipeline that takes raw news text and predicts which of these four categories it belongs to.

### What you will implement

You will complete **five exercises** that mirror a real-world fine-tuning workflow:

1. A custom PyTorch `Dataset` class that tokenizes text on the fly
2. A data collator for dynamic batching
3. `DataLoader` objects for training and validation
4. Partial model freezing for efficient fine-tuning
5. Hyperparameter selection to hit target performance

> **Before you start:** make sure you have worked through the four lab notebooks in this module, especially Lab 4 (fine-tuning DistilBERT). The concepts and code patterns there are the foundation for everything here.

## Table of Contents

- [Imports](#imports)
- [Device Setup](#device-setup)
- [Load and Explore the Data](#load-explore)
- [Load the Pre-trained Model (safetensors)](#load-model)
- [Exercise 1 — Custom Dataset Class](#ex-1)
- [Exercise 2 — Data Collator](#ex-2)
- [Exercise 3 — DataLoaders](#ex-3)
- [Exercise 4 — Partial Model Freezing](#ex-4)
- [Exercise 5 — Train the Model](#ex-5)
- [Evaluation](#evaluation)
- [Bonus — Try a Different Pre-trained Model](#bonus)

## Imports

In [28]:
!pip install torchmetrics

After installing `torchmetrics`, please re-run the `Imports` cell (`cell-imports`) to ensure all modules are loaded correctly.

In [32]:
import torch
from datasets import load_dataset
from torch.utils.data import DataLoader, Dataset
from transformers import DataCollatorWithPadding

import homework_helper_utils

### Uploading `homework_helper_utils.py`

To resolve the `ModuleNotFoundError`, you need to upload the `homework_helper_utils.py` file to your Colab environment. You can do this by running the following code cell, which will prompt you to select the file from your local machine. Alternatively, you can drag and drop the file into the file browser on the left-hand side of Colab.

In [31]:
from google.colab import files

print("Please upload the 'homework_helper_utils.py' file.")
uploaded = files.upload()

Please upload the 'homework_helper_utils.py' file.


Saving homework_helper_utils.py to homework_helper_utils (1).py


After uploading, you may need to re-run the `cell-imports` cell to ensure the module is correctly loaded.

## Device Setup

Training on a GPU is significantly faster. The cell below automatically selects a GPU if one is available, and falls back to CPU otherwise.

In [33]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## Load and Explore the Data

AG News is provided through the Hugging Face `datasets` library. It comes pre-split into a training set (120,000 articles) and a test set (7,600 articles).

To keep training times manageable in this lab environment we will work with a **subset of 20,000 articles** from the training split, which we then divide into our own train/validation sets. The dataset is already class-balanced, so a random subset preserves that balance.

In [34]:
# Load AG News from the Hugging Face datasets hub (cached locally after first download)
raw_dataset = load_dataset("ag_news")

# Use a 20,000-sample subset for a manageable training time
NUM_SAMPLES = 20_000
subset = raw_dataset["train"].shuffle(seed=79).select(range(NUM_SAMPLES))

texts  = subset["text"]
labels = subset["label"]

print(f"Loaded {len(texts)} samples")
print(f"\nExample article:\n  Text : {texts[0][:120]}...")
print(f"  Label: {labels[0]}")

Loaded 20000 samples

Example article:
  Text : Internet  #39;overlay #39; could boost performance The internet could be made more secure, reliable and powerful by over...
  Label: 3


In [35]:
# Map integer labels to human-readable category names
id2cat = {0: "World", 1: "Sports", 2: "Business", 3: "Sci/Tech"}
cat2id = {v: k for k, v in id2cat.items()}

# Check the class distribution in our subset
import collections
counts = collections.Counter(labels)
print("Class distribution:")
for label_id, count in sorted(counts.items()):
    print(f"  {label_id} ({id2cat[label_id]:<10}): {count} samples ({count/NUM_SAMPLES:.1%})")

Class distribution:
  0 (World     ): 5003 samples (25.0%)
  1 (Sports    ): 5031 samples (25.2%)
  2 (Business  ): 5019 samples (25.1%)
  3 (Sci/Tech  ): 4947 samples (24.7%)


#### Expected output

With `NUM_SAMPLES = 20_000` and `seed=42` you should see approximately equal class counts:

```
Label distribution: Counter({0: 5000, 1: 5002, 2: 4999, 3: 4999})
```

The exact numbers will vary slightly with different seeds, but the four classes should each represent roughly 25 % of the 20 000 sample subset.

## Load the Pre-trained Model

### A note on model formats: safetensors vs. pickle

When you download a pre-trained model, it can come in two formats:

| Format | File | How it's loaded | Risk |
|--------|------|-----------------|------|
| **Legacy** | `pytorch_model.bin` | `torch.load()` which uses Python `pickle` | ⚠️ Pickle can execute arbitrary code on load — a malicious model file is a backdoor |
| **Safetensors** | `model.safetensors` | Zero-copy memory-mapped read, no code execution | ✅ Safe to load from any source |

HuggingFace now ships all modern models in safetensors by default. Passing `use_safetensors=True` to `from_pretrained` makes this explicit.

The cell below downloads DistilBERT from the HuggingFace Hub (files are cached locally after the first run) and attaches a fresh **4-class** classification head. It returns both the model and the tokenizer — the tokenizer will be needed throughout the pipeline.

In [49]:
MODEL_ID = "albert-base-v2"
NUM_CLASSES = 4

model, tokenizer = homework_helper_utils.load_bert(MODEL_ID, num_classes=NUM_CLASSES)

Loading 'albert-base-v2' from the HuggingFace Hub (safetensors format)...


config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/760k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/47.4M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/25 [00:00<?, ?it/s]

AlbertForSequenceClassification LOAD REPORT from: albert-base-v2
Key                          | Status     | 
-----------------------------+------------+-
predictions.decoder.bias     | UNEXPECTED | 
predictions.dense.bias       | UNEXPECTED | 
predictions.dense.weight     | UNEXPECTED | 
predictions.LayerNorm.bias   | UNEXPECTED | 
predictions.bias             | UNEXPECTED | 
predictions.LayerNorm.weight | UNEXPECTED | 
classifier.bias              | MISSING    | 
classifier.weight            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded successfully (11,686,660 parameters).


In [37]:
# Bonus: swap MODEL_ID here and re-run from "Load the Pre-trained Model" onwards
# MODEL_ID = "distilbert-base-multilingual-cased"
# MODEL_ID = "albert-base-v2"
# MODEL_ID = "roberta-base"

---
## Exercise 1 — Custom Dataset Class

A PyTorch `Dataset` is the foundation of any training pipeline. It defines how individual samples are retrieved and preprocessed.

Your task is to implement `AGNewsDataset`, a `Dataset` subclass that:

1. Stores the raw texts, integer labels, and tokenizer.
2. Returns the total number of samples via `__len__`.
3. For a given index, tokenizes the corresponding text in `__getitem__` and attaches the label.

### Key implementation notes

- Call the tokenizer with `truncation=True` and `max_length=512`. Do **not** add padding here — that is handled later by the data collator.
- The label must be added to the encoding dictionary as a `torch.long` tensor with key `"labels"`.
- Return the encoding dictionary directly from `__getitem__` — the collator expects this format.

### Why no padding here?

Padding all sequences to the same length during dataset creation would waste computation, because sequences within a batch would be padded to the *global* maximum length. Instead, we pad dynamically per batch (in Exercise 2), so each batch is only as long as its longest sequence.

In [50]:
class AGNewsDataset(Dataset):
    """
    A PyTorch Dataset for AG News topic classification.

    Tokenizes text on the fly and returns a dictionary compatible with
    Hugging Face DataCollatorWithPadding.

    Args:
        texts (list[str])  : Raw news article strings.
        labels (list[int]) : Integer class labels (0–3).
        tokenizer          : A Hugging Face tokenizer.
    """

    def __init__(self, texts, labels, tokenizer):
        ### START CODE HERE ###

        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer

        ### END CODE HERE ###

    def __len__(self):
        ### START CODE HERE ###

        return len(self.labels)

        ### END CODE HERE ###

    def __getitem__(self, idx):
        """
        Tokenizes the text at position `idx` and returns the encoding with
        the label added as a torch.long tensor under the key 'labels'.
        """
        ### START CODE HERE ###
        text = self.texts[idx]
        input = tokenizer(text)
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding=False,
            return_tensors="pt"
        )
        encoding = {key: val.squeeze(0) for key, val in encoding.items()}
        encoding["labels"]= torch.tensor(self.labels[idx], dtype=torch.long)

        return encoding

        ### END CODE HERE ###

### Build and split the dataset

Now that the class is defined, we create the full dataset and split it into 80 % training and 20 % validation using `homework_helper_utils.create_dataset_splits`.

In [51]:
full_dataset = AGNewsDataset(texts, labels, tokenizer)

train_dataset, val_dataset = homework_helper_utils.create_dataset_splits(full_dataset, train_split_percentage=0.8)

print(f"Training samples  : {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")

Training samples  : 16000
Validation samples: 4000


#### Expected output

```
Training samples  : 16000
Validation samples: 4000
```

---
## Exercise 2 — Data Collator

The data collator is responsible for taking a list of individual samples from the `Dataset` and assembling them into a single batch. A key task is **padding** — ensuring all sequences in a batch have the same length so they can be stacked into a tensor.

Hugging Face's `DataCollatorWithPadding` does this automatically: it pads sequences within each batch to the length of the longest sequence in *that batch* only. This dynamic padding is more efficient than padding everything to the global maximum.

Implement `create_data_collator` to return a configured `DataCollatorWithPadding` instance.

In [52]:
def create_data_collator(tokenizer):
    """
    Creates a DataCollatorWithPadding for dynamic per-batch padding.

    Args:
        tokenizer: The Hugging Face tokenizer used to tokenize the data.

    Returns:
        A DataCollatorWithPadding instance configured with the given tokenizer.
    """
    ### START CODE HERE ###

    return DataCollatorWithPadding(tokenizer=tokenizer)

    ### END CODE HERE ###

In [53]:
data_collator = create_data_collator(tokenizer)
print(f"Collator type: {type(data_collator).__name__}")

Collator type: DataCollatorWithPadding


#### Expected output

```
Collator type: DataCollatorWithPadding
```

---
## Exercise 3 — DataLoaders

PyTorch `DataLoader` objects wrap a `Dataset` to provide batched, shuffled, and parallel data loading during training.

Implement `create_dataloaders` to return two DataLoaders:

- **`train_loader`**: shuffles the data at each epoch (crucial to prevent the model from memorising the order of samples)
- **`val_loader`**: does *not* shuffle — consistent order makes validation results reproducible

Both loaders must use the `collate_fn` you created in Exercise 2 to perform dynamic padding.

In [54]:
def create_dataloaders(train_dataset, val_dataset, batch_size, collate_fn):
    """
    Creates training and validation DataLoaders.

    Args:
        train_dataset : PyTorch Dataset for training.
        val_dataset   : PyTorch Dataset for validation.
        batch_size    : Number of samples per batch.
        collate_fn    : Function to collate individual samples into batches
                        (use the DataCollatorWithPadding from Exercise 2).

    Returns:
        A tuple (train_loader, val_loader) of DataLoader objects.
    """
    ### START CODE HERE ###

    train_loader=DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=collate_fn
    )

    val_loader=DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collate_fn
    )
    return train_loader, val_loader
    ### END CODE HERE ###

In [55]:
BATCH_SIZE = 32

train_loader, val_loader = create_dataloaders(
    train_dataset, val_dataset, BATCH_SIZE, data_collator
)

print(f"Training batches  : {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")

# Inspect one batch to verify the shapes and keys
sample_batch = next(iter(train_loader))
print(f"\nBatch keys  : {list(sample_batch.keys())}")
print(f"input_ids   : {sample_batch['input_ids'].shape}")
print(f"attention_mask: {sample_batch['attention_mask'].shape}")
print(f"labels      : {sample_batch['labels'].shape}")

Training batches  : 500
Validation batches: 125

Batch keys  : ['input_ids', 'attention_mask', 'labels']
input_ids   : torch.Size([32, 137])
attention_mask: torch.Size([32, 137])
labels      : torch.Size([32])


#### Expected output

The batch shapes will vary per batch because `DataCollatorWithPadding` pads only to the longest sequence in each batch (dynamic padding). You should see something like:

```
Batch shapes:
  input_ids     : torch.Size([32, <variable>])
  attention_mask: torch.Size([32, <variable>])
  labels        : torch.Size([32])
```

`input_ids` and `attention_mask` will both be `[32, L]` where `L` is the length of the longest sequence in that specific batch. `labels` is always `[batch_size]`.

---
## Exercise 4 — Partial Model Freezing

DistilBERT has **6 transformer layers** and roughly 66 million parameters. Training all of them from scratch would:
- Take a very long time
- Risk catastrophic forgetting of the language understanding learned during pre-training
- Require far more data to generalise well

**Partial fine-tuning** (also called parameter-efficient fine-tuning) addresses this by *freezing* most of the model and only updating a small fraction of the weights:

1. **Freeze all parameters** — set `requires_grad = False` for every parameter.
2. **Unfreeze the last `layers_to_train` transformer layers** — the later layers are most task-specific and benefit most from fine-tuning.
3. **Unfreeze the classification head** — `pre_classifier` and `classifier` are new and must be trained.

### DistilBERT layer access pattern

```python
model.distilbert.transformer.layer   # ModuleList of 6 layers (index 0 = earliest, 5 = latest)
model.pre_classifier                 # Linear layer before the final output
model.classifier                     # Final Linear layer (outputs logits)
```

To unfreeze the last `layers_to_train` layers you can use negative indexing:
```python
model.distilbert.transformer.layer[-(i+1)]  # i=0 → last layer, i=1 → second-to-last, ...
```

In [56]:
def partially_freeze_bert_layers(model, layers_to_train=3):
    """
    Freezes all model parameters, then selectively unfreezes the last
    `layers_to_train` transformer layers and the classification head.

    Args:
        model          : A DistilBertForSequenceClassification model.
        layers_to_train: Number of transformer layers (from the end) to unfreeze.
                         Must be between 1 and 6.
    """
    ### START CODE HERE ###

    # Step 1: Freeze all parameters
    for param in model.parameters():
      param.requires_grad= False
    # Step 2: Unfreeze the last `layers_to_train` transformer layers
    # The original code was trying to iterate over a single TransformerBlock.
    # We need to iterate over a slice of the ModuleList for the last 'layers_to_train' layers.
    for layer in model.distilbert.transformer.layer[-layers_to_train:]:
      for param in layer.parameters():
        param.requires_grad=True
    # Step 3: Unfreeze the classification head (pre_classifier and classifier)
    for param in model.pre_classifier.parameters():
        param.requires_grad=True
    for param in model.classifier.parameters():
        param.requires_grad=True

    ### END CODE HERE ###

### Inspect trainable parameters

After applying the freezing, let's verify how many parameters will actually be updated during training. This is a useful sanity check.

In [57]:
# We'll apply the freeze once we've chosen layers_to_train in Exercise 6.
# For now, just verify the function is correct with a preview.
import copy
model_preview = copy.deepcopy(model)
partially_freeze_bert_layers(model_preview, layers_to_train=3)

total_params   = sum(p.numel() for p in model_preview.parameters())
trainable_params = sum(p.numel() for p in model_preview.parameters() if p.requires_grad)
print(f"Total parameters    : {total_params:,}")
print(f"Trainable parameters: {trainable_params:,} ({trainable_params/total_params:.1%} of total)")
del model_preview

AttributeError: 'AlbertForSequenceClassification' object has no attribute 'distilbert'

#### Expected output (with `layers_to_train=3`)

```
Total parameters    : 66,955,780
Trainable parameters: 21,383,172 (31.9% of total)
```

With `layers_to_train=3` approximately one third of the 66 million parameters are trainable. Changing `layers_to_train` between 1 and 6 changes this fraction proportionally.

---
## Exercise 5 — Train the Model

You have all the pieces in place. Now it is time to train!

Your task is to choose three hyperparameters that together allow the model to meet the target performance:

| Hyperparameter    | Description                                              | Suggested range |
|-------------------|----------------------------------------------------------|-----------------|
| `layers_to_train` | Number of DistilBERT transformer layers to unfreeze      | 2 – 6           |
| `learning_rate`   | Step size for AdamW optimizer                            | 1e-5 – 5e-4     |
| `num_epochs`      | Number of full passes over the training data             | 3 – 10          |

### Tips

- A good starting point: `layers_to_train=4`, `learning_rate=5e-5`, `num_epochs=5`.
- More unfrozen layers generally improve accuracy but increase training time.
- If the model is not learning (loss stays flat), try a slightly higher learning rate.
- If the model oscillates or diverges, lower the learning rate.

In [46]:
### START CODE HERE ###

layers_to_train = 4   # int, 1–6
learning_rate   = 5e-5   # float, e.g. 5e-5
num_epochs      = 5   # int, >= 2

### END CODE HERE ###

In [47]:
# Apply partial freezing with your chosen value
partially_freeze_bert_layers(model, layers_to_train=layers_to_train)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Training {trainable_params:,} / {total_params:,} parameters ({trainable_params/total_params:.1%})")

# Standard cross-entropy loss — AG News is well balanced so no weighting needed
loss_function = torch.nn.CrossEntropyLoss()

Training 28,945,156 / 135,327,748 parameters (21.4%)


In [48]:
# Run the training and validation loop
model, final_results = homework_helper_utils.training_loop(
    model        = model,
    train_loader = train_loader,
    val_loader   = val_loader,
    loss_function= loss_function,
    learning_rate= learning_rate,
    num_epochs   = num_epochs,
    device       = device
)

print(f"\nFinal val loss    : {final_results['val_loss']:.4f}")
print(f"Final val accuracy: {final_results['val_accuracy']:.4f}")
print(f"Final val F1      : {final_results['val_f1']:.4f}")

Training Progress:   0%|          | 0/5 [00:00<?, ?it/s]

Epoch 1/5 Training:   0%|          | 0/500 [00:00<?, ?it/s]

Epoch 1/5 Validation:   0%|          | 0/125 [00:00<?, ?it/s]

Epoch 1 Metrics -> Val Loss: 0.3170, Val Acc: 0.8817, Val F1: 0.8825


Epoch 2/5 Training:   0%|          | 0/500 [00:00<?, ?it/s]

...

Epoch 4/5 Validation:   0%|          | 0/125 [00:00<?, ?it/s]

Epoch 4 Metrics -> Val Loss: 0.2634, Val Acc: 0.9162, Val F1: 0.9162


Epoch 5/5 Training:   0%|          | 0/500 [00:00<?, ?it/s]

Epoch 5/5 Validation:   0%|          | 0/125 [00:00<?, ?it/s]

Epoch 5 Metrics -> Val Loss: 0.2981, Val Acc: 0.9107, Val F1: 0.9114

--- Training complete ---

Final val loss    : 0.2981
Final val accuracy: 0.9107
Final val F1      : 0.9114


#### Expected output

With the suggested hyperparameters (`layers_to_train=4`, `learning_rate=5e-5`, `num_epochs=5`) you should reach:

```
Final val loss    : < 0.30
Final val accuracy: 0.91 – 0.94
Final val F1      : ≥ 0.91
```

Training 5 epochs on CPU takes approximately 1–2 hours; on a GPU it takes 5–10 minutes. If your metrics are significantly lower, try adjusting the learning rate or increasing `layers_to_train`.

---
## Evaluation

Let's take a closer look at *how* the model performs across the four categories — not just the average score.

### Save your fine-tuned model (safetensors)

`save_pretrained` with `safe_serialization=True` writes the model weights as a `.safetensors` file — never pickle. You can reload it later with `AutoModelForSequenceClassification.from_pretrained("./agnews-distilbert", use_safetensors=True)`.

In [ ]:
# Save the fine-tuned model in safetensors format (no pickle, no arbitrary code execution)
model.save_pretrained("./agnews-distilbert", safe_serialization=True)
tokenizer.save_pretrained("./agnews-distilbert")
print("Model saved to ./agnews-distilbert/")

In [ ]:
homework_helper_utils.analyze_and_plot_results(final_results, id2cat)

### Try it on your own news headlines

Use the cell below to pass any text through your trained classifier and see which category it predicts.

In [ ]:
example_headlines = [
    "NASA's Perseverance rover discovers ancient river delta on Mars",
    "Federal Reserve raises interest rates amid inflation concerns",
    "Manchester City wins the Premier League title for the fourth time",
    "United Nations Security Council meets to discuss ceasefire resolution",
]

print("Predictions:")
for headline in example_headlines:
    prediction = homework_helper_utils.predict_category(model, tokenizer, headline, device, id2cat)
    print(f"  [{prediction:<10}] {headline}")

#### Expected predictions

```
Predictions:
  [Sci/Tech  ] NASA's Perseverance rover discovers ancient river delta on Mars
  [Business  ] Federal Reserve raises interest rates amid inflation concerns
  [Sports    ] Manchester City wins the Premier League title for the fourth time
  [World     ] United Nations Security Council meets to discuss ceasefire resolution
```

Try adding your own headlines — news from today's front page works well!

---
## Conclusion

Congratulations on completing the assignment!

You have built a complete fine-tuning pipeline from scratch:

| Step | What you built |
|------|----------------|
| Dataset | `AGNewsDataset` — on-the-fly tokenization, no upfront padding |
| Collation | `DataCollatorWithPadding` — efficient per-batch dynamic padding |
| DataLoaders | Shuffled training, sequential validation |
| Partial freezing | Only the top transformer layers + head are updated |
| Training | AdamW, torchmetrics, tqdm — full monitoring loop |
| Safe persistence | Model saved in safetensors format (no pickle) |

The same pattern — pretrained backbone + frozen lower layers + fine-tuned upper layers + task-specific head — is the workhorse of modern NLP and increasingly of computer vision as well. You now have hands-on experience with the full workflow.

---
## Bonus — Try a Different Pre-trained Model

One of the great advantages of building a Hub-based pipeline is that swapping the backbone model is a **single-line change**: just set `MODEL_ID` to a different Hub identifier and re-run from the "Load the Pre-trained Model" cell onwards.

Here are a few models worth experimenting with:

| Model ID | Description | Expected difference |
|---|---|---|
| `distilbert-base-uncased` | The baseline (40 % smaller than BERT-base) | — baseline — |
| `distilbert-base-multilingual-cased` | Multilingual DistilBERT — 104 languages | Similar size; try classifying non-English news |
| `albert-base-v2` | Parameter-sharing architecture, ~12M params | Smaller and faster but often slightly lower accuracy |
| `roberta-base` | RoBERTa — improved BERT training procedure | Often 1–2 % higher accuracy at the cost of longer training |

> **Hint:** `roberta-base` uses a different tokenizer and a different model attribute path for layers (`model.roberta.encoder.layer[i]` instead of `model.distilbert.transformer.layer[i]`). You will need to update `partially_freeze_bert_layers` accordingly — reading the model card at [huggingface.co/roberta-base](https://huggingface.co/roberta-base) will tell you the architecture.

**Challenge:** Run the pipeline with two different models and compare their final validation F1 scores. Which model gives the best result for AG News? Which is fastest to train?

In [ ]:
# Bonus: swap MODEL_ID here and re-run from "Load the Pre-trained Model" onwards
# MODEL_ID = "distilbert-base-multilingual-cased"
# MODEL_ID = "albert-base-v2"
# MODEL_ID = "roberta-base"